# Shape-aware YouTube-ASL contrastive pretraining

Run all cells after selecting a GPU runtime. This is a new experiment: explicit body and hand graphs are encoded by part-specific ST-GCNs, followed by the same 6-layer temporal Transformer and sentence-level contrastive objective. It uses no translations, glosses, lexical boundaries, or latent units. Each official ZIP shard is downloaded once, trained as one epoch, and deleted after a resumable checkpoint-and-metrics ZIP is sent to your browser.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

if shutil.which('aria2c') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)

REPO_URL = 'https://github.com/ss-sebastian/youtube-asl-skeleton-bert.git'
REPO_REF = 'agent/shape-aware-stgcn'
PROJECT = Path('/content/youtube-asl-skeleton-bert')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
print('Shape-aware project ready:', PROJECT)


## Optional resume
If Colab disconnected after a completed shard, upload the latest downloaded ZIP into `/content` before running the next cell. Accepted names are `youtube_asl_shape_contrastive_resume_after_shard_XX.zip` or `youtube_asl_shape_contrastive_full_resume.zip`. Otherwise, continue directly.


In [ ]:
import runpy
runpy.run_path(str(PROJECT / 'scripts/colab_shape_full_train.py'), run_name='__main__')


In [ ]:
from IPython.display import display
import pandas as pd

checkpoint_root = Path('/content/sign_semantics_youtube_asl/full_shape_contrastive/checkpoints')
metrics_path = checkpoint_root / 'metrics.jsonl'
assert (checkpoint_root / 'last.pt').exists(), 'Training did not produce last.pt'
metrics = pd.read_json(metrics_path, lines=True)
display(metrics)
metrics.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', grid=True)
